# Trigger-Based Strategy: Buy Low, Sell High Only

**Concept:** Only act on HIGH CONVICTION signals:
- 🟢 **Aggressive BUY** when composite score < -1.0 (deep value)
- 🔴 **Aggressive SELL** when composite score > 1.5 (euphoria)
- 🟡 **HOLD** everything in between

This avoids constant position adjustments and only trades at extremes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None:
            df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

data = {m: load_metric(m) for m in ['price', 'mvrv', 'mvrv_sth', 'mvrv_lth', 'nupl', 'sopr', 'aviv']}
print("Data loaded")

In [ ]:
# Build composite score (same as before)
CONFIG = {
    'mvrv': {'weight': 0.30, 'bullish': 1.0, 'bearish': 2.4},
    'mvrv_sth': {'weight': 0.15, 'bullish': 1.0, 'bearish': 1.4},
    'mvrv_lth': {'weight': 0.15, 'bullish': 1.5, 'bearish': 3.5},
    'nupl': {'weight': 0.20, 'bullish': 0.25, 'bearish': 0.6},
    'sopr': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.05},
    'aviv': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.5},
}

def score_metric(value, bullish, bearish):
    if pd.isna(value): return 0
    midpoint = (bullish + bearish) / 2
    if value <= bullish:
        return -1 - (bullish - value) / bullish
    elif value <= midpoint:
        return -1 + (value - bullish) / (midpoint - bullish)
    elif value <= bearish:
        return (value - midpoint) / (bearish - midpoint)
    else:
        return 1 + min((value - bearish) / bearish, 1)

df = data['price'][['value']].rename(columns={'value': 'price'}).copy()
df['returns'] = df['price'].pct_change()

for metric in CONFIG.keys():
    if metric in data and not data[metric].empty:
        df = df.join(data[metric][['value']].rename(columns={'value': metric}), how='left')
        df[metric] = df[metric].ffill()

def calc_composite(row):
    total_score, total_weight = 0, 0
    for metric, cfg in CONFIG.items():
        if metric in row and pd.notna(row[metric]):
            total_score += score_metric(row[metric], cfg['bullish'], cfg['bearish']) * cfg['weight']
            total_weight += cfg['weight']
    return total_score / total_weight if total_weight > 0 else 0

df['composite'] = df.apply(calc_composite, axis=1)
print(f"Data: {df.index[0].date()} to {df.index[-1].date()}")

## Trigger-Based Position Management

**Rules:**
1. Start at 50% position (or user-defined)
2. When score drops below BUY threshold → Go to 100%
3. When score rises above SELL threshold → Go to minimum (e.g., 20%)
4. Otherwise → HOLD current position (no changes)

In [ ]:
def trigger_strategy(df, buy_threshold=-1.0, sell_threshold=1.5, 
                     buy_position=1.0, sell_position=0.20, start_position=0.5):
    """
    Trigger-based strategy:
    - BUY (go to buy_position) when score < buy_threshold
    - SELL (go to sell_position) when score > sell_threshold
    - HOLD current position otherwise
    """
    positions = [start_position]
    signals = ['START']
    
    for i in range(1, len(df)):
        score = df['composite'].iloc[i-1]  # Use previous day's score
        current_pos = positions[-1]
        
        if score < buy_threshold:
            positions.append(buy_position)
            signals.append('BUY' if current_pos < buy_position else 'HOLD')
        elif score > sell_threshold:
            positions.append(sell_position)
            signals.append('SELL' if current_pos > sell_position else 'HOLD')
        else:
            positions.append(current_pos)  # HOLD
            signals.append('HOLD')
    
    return positions, signals

# Test different threshold combinations
threshold_configs = [
    {'name': 'Conservative', 'buy': -1.5, 'sell': 2.0, 'sell_pos': 0.30},
    {'name': 'Moderate', 'buy': -1.0, 'sell': 1.5, 'sell_pos': 0.25},
    {'name': 'Aggressive', 'buy': -0.75, 'sell': 1.25, 'sell_pos': 0.20},
    {'name': 'Very Aggressive', 'buy': -0.5, 'sell': 1.0, 'sell_pos': 0.15},
]

print("Threshold configurations defined")

In [ ]:
# Backtest all configurations
def backtest_trigger(df, config):
    positions, signals = trigger_strategy(
        df, 
        buy_threshold=config['buy'], 
        sell_threshold=config['sell'],
        sell_position=config['sell_pos']
    )
    
    bt = df[['price', 'returns', 'composite']].copy()
    bt['position'] = positions
    bt['signal'] = signals
    bt['strat_returns'] = bt['position'] * bt['returns']
    bt['equity'] = 100000 * (1 + bt['strat_returns']).cumprod()
    bt['dd'] = bt['equity'] / bt['equity'].cummax() - 1
    
    # Count trades
    trades = sum(1 for s in signals if s in ['BUY', 'SELL'])
    buys = sum(1 for s in signals if s == 'BUY')
    sells = sum(1 for s in signals if s == 'SELL')
    
    years = (bt.index[-1] - bt.index[0]).days / 365
    
    return {
        'name': config['name'],
        'buy_thresh': config['buy'],
        'sell_thresh': config['sell'],
        'total_return': (bt['equity'].iloc[-1] / 100000 - 1) * 100,
        'cagr': ((bt['equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100,
        'max_dd': bt['dd'].min() * 100,
        'sharpe': (bt['strat_returns'].mean() / bt['strat_returns'].std()) * np.sqrt(365),
        'trades': trades,
        'buys': buys,
        'sells': sells,
        'avg_position': bt['position'].mean(),
        'equity_curve': bt['equity'],
        'position_curve': bt['position'],
        'signal_series': bt['signal'],
        'dd_curve': bt['dd']
    }

# HODL baseline
hodl_equity = 100000 * (1 + df['returns']).cumprod()
hodl_dd = hodl_equity / hodl_equity.cummax() - 1
years = (df.index[-1] - df.index[0]).days / 365

results = []
for config in threshold_configs:
    results.append(backtest_trigger(df, config))

print("Backtests complete")

In [ ]:
# Results comparison
print("\n" + "="*100)
print("TRIGGER STRATEGY COMPARISON")
print("="*100)
print(f"Period: {df.index[0].date()} to {df.index[-1].date()} ({years:.1f} years)")

# HODL stats
hodl_return = (hodl_equity.iloc[-1] / 100000 - 1) * 100
hodl_cagr = ((hodl_equity.iloc[-1] / 100000) ** (1/years) - 1) * 100
hodl_max_dd = hodl_dd.min() * 100
hodl_sharpe = (df['returns'].mean() / df['returns'].std()) * np.sqrt(365)

print(f"\n{'Strategy':<18} {'Buy<':>6} {'Sell>':>6} {'Return':>10} {'CAGR':>8} {'MaxDD':>8} {'Sharpe':>7} {'Trades':>7} {'AvgPos':>7}")
print("-"*100)
print(f"{'HODL':<18} {'--':>6} {'--':>6} {hodl_return:>9.0f}% {hodl_cagr:>7.0f}% {hodl_max_dd:>7.0f}% {hodl_sharpe:>7.2f} {'--':>7} {'100%':>7}")
print("-"*100)

for r in results:
    print(f"{r['name']:<18} {r['buy_thresh']:>6.2f} {r['sell_thresh']:>6.2f} "
          f"{r['total_return']:>9.0f}% {r['cagr']:>7.0f}% {r['max_dd']:>7.0f}% "
          f"{r['sharpe']:>7.2f} {r['trades']:>7} {r['avg_position']:>6.0%}")

print("-"*100)

In [ ]:
# Visualize equity curves
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Equity curves
axes[0].semilogy(df.index, hodl_equity, 'orange', linewidth=2, label=f'HODL ({hodl_cagr:.0f}% CAGR)', alpha=0.7)
colors = ['#22c55e', '#3b82f6', '#a855f7', '#ef4444']
for r, c in zip(results, colors):
    axes[0].semilogy(r['equity_curve'].index, r['equity_curve'], color=c, linewidth=2,
                     label=f"{r['name']} ({r['cagr']:.0f}% CAGR)")

axes[0].set_ylabel('Equity ($)')
axes[0].set_title('Trigger Strategy: Buy Low, Sell High Only', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Position over time (using Moderate strategy)
mod_result = results[1]  # Moderate
axes[1].fill_between(mod_result['position_curve'].index, 0, mod_result['position_curve']*100, 
                     color='#3b82f6', alpha=0.5)
axes[1].set_ylabel('Position (%)')
axes[1].set_ylim(0, 105)
axes[1].axhline(y=100, color='#22c55e', linestyle='--', alpha=0.5, label='Full (100%)')
axes[1].axhline(y=25, color='#ef4444', linestyle='--', alpha=0.5, label='Minimum (25%)')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Composite score with thresholds
axes[2].fill_between(df.index, df['composite'], 0, 
                     where=df['composite'] < 0, color='#22c55e', alpha=0.5)
axes[2].fill_between(df.index, df['composite'], 0, 
                     where=df['composite'] >= 0, color='#ef4444', alpha=0.5)
axes[2].axhline(y=-1.0, color='#22c55e', linestyle='--', linewidth=2, label='Buy threshold')
axes[2].axhline(y=1.5, color='#ef4444', linestyle='--', linewidth=2, label='Sell threshold')
axes[2].set_ylabel('Composite Score')
axes[2].set_xlabel('Date')
axes[2].set_ylim(-2.5, 2.5)
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Trade Analysis: When Did We Buy/Sell?

In [ ]:
# Analyze the Moderate strategy trades
mod = results[1]
bt = df.copy()
bt['position'] = mod['position_curve']
bt['signal'] = mod['signal_series']

# Find buy and sell dates
buy_dates = bt[bt['signal'] == 'BUY'].index
sell_dates = bt[bt['signal'] == 'SELL'].index

print("\n" + "="*70)
print("TRADE LOG (Moderate Strategy)")
print("="*70)

print(f"\n🟢 BUY SIGNALS ({len(buy_dates)} total):")
print("-"*50)
for date in buy_dates[:20]:  # First 20
    price = bt.loc[date, 'price']
    score = bt.loc[date, 'composite']
    print(f"  {date.date()}  ${price:>10,.0f}  Score: {score:+.2f}")
if len(buy_dates) > 20:
    print(f"  ... and {len(buy_dates) - 20} more")

print(f"\n🔴 SELL SIGNALS ({len(sell_dates)} total):")
print("-"*50)
for date in sell_dates[:20]:  # First 20
    price = bt.loc[date, 'price']
    score = bt.loc[date, 'composite']
    print(f"  {date.date()}  ${price:>10,.0f}  Score: {score:+.2f}")
if len(sell_dates) > 20:
    print(f"  ... and {len(sell_dates) - 20} more")

In [ ]:
# Visualize buy/sell timing on price chart
fig, ax = plt.subplots(figsize=(14, 8))

ax.semilogy(df.index, df['price'], color='white', linewidth=1, alpha=0.7)

# Mark buy signals
ax.scatter(buy_dates, df.loc[buy_dates, 'price'], 
           color='#22c55e', marker='^', s=100, label=f'BUY ({len(buy_dates)})', zorder=5)

# Mark sell signals  
ax.scatter(sell_dates, df.loc[sell_dates, 'price'],
           color='#ef4444', marker='v', s=100, label=f'SELL ({len(sell_dates)})', zorder=5)

ax.set_ylabel('Price (USD)')
ax.set_xlabel('Date')
ax.set_title('Trigger Strategy: Buy & Sell Points (Moderate)', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cycle-by-Cycle Analysis

In [ ]:
# Define market cycles
cycles = [
    ('2015 Bottom', '2015-01-01', '2015-12-31'),
    ('2017 Bull', '2016-01-01', '2017-12-31'),
    ('2018 Bear', '2018-01-01', '2018-12-31'),
    ('2019-2020', '2019-01-01', '2020-03-31'),
    ('2020-2021 Bull', '2020-04-01', '2021-11-30'),
    ('2022 Bear', '2021-12-01', '2022-12-31'),
    ('2023-2024', '2023-01-01', '2024-12-31'),
    ('2025+', '2025-01-01', '2026-12-31'),
]

print("\nCYCLE-BY-CYCLE PERFORMANCE")
print("="*90)
print(f"{'Cycle':<15} {'HODL':>12} {'Trigger':>12} {'Diff':>10} {'Buys':>6} {'Sells':>6} {'AvgPos':>8}")
print("-"*90)

for name, start, end in cycles:
    try:
        mask = (df.index >= start) & (df.index <= end)
        if mask.sum() == 0:
            continue
            
        cycle_df = df[mask]
        
        # HODL return
        hodl_ret = (cycle_df['price'].iloc[-1] / cycle_df['price'].iloc[0] - 1) * 100
        
        # Trigger return
        cycle_pos = mod['position_curve'][mask]
        cycle_returns = cycle_df['returns'] * cycle_pos
        trig_ret = ((1 + cycle_returns).cumprod().iloc[-1] - 1) * 100
        
        # Trades in this cycle
        cycle_signals = mod['signal_series'][mask]
        buys = (cycle_signals == 'BUY').sum()
        sells = (cycle_signals == 'SELL').sum()
        avg_pos = cycle_pos.mean()
        
        diff = trig_ret - hodl_ret
        print(f"{name:<15} {hodl_ret:>+11.0f}% {trig_ret:>+11.0f}% {diff:>+9.0f}% {buys:>6} {sells:>6} {avg_pos:>7.0%}")
    except:
        pass

print("-"*90)

## Optimal Threshold Finder

In [ ]:
# Grid search for optimal thresholds
buy_range = np.arange(-2.0, -0.25, 0.25)
sell_range = np.arange(0.5, 2.25, 0.25)

grid_results = []

for buy_thresh in buy_range:
    for sell_thresh in sell_range:
        if sell_thresh <= buy_thresh + 0.5:  # Need gap between buy and sell
            continue
            
        config = {'name': f'B{buy_thresh:.1f}_S{sell_thresh:.1f}', 
                  'buy': buy_thresh, 'sell': sell_thresh, 'sell_pos': 0.25}
        result = backtest_trigger(df, config)
        grid_results.append(result)

# Find best by different metrics
best_return = max(grid_results, key=lambda x: x['total_return'])
best_sharpe = max(grid_results, key=lambda x: x['sharpe'])
best_dd = max(grid_results, key=lambda x: x['max_dd'])  # Less negative = better

print("\nOPTIMAL THRESHOLDS")
print("="*70)
print(f"\n📈 Best Return: Buy < {best_return['buy_thresh']:.2f}, Sell > {best_return['sell_thresh']:.2f}")
print(f"   Return: {best_return['total_return']:.0f}%, CAGR: {best_return['cagr']:.0f}%, MaxDD: {best_return['max_dd']:.0f}%")

print(f"\n📊 Best Sharpe: Buy < {best_sharpe['buy_thresh']:.2f}, Sell > {best_sharpe['sell_thresh']:.2f}")
print(f"   Sharpe: {best_sharpe['sharpe']:.2f}, Return: {best_sharpe['total_return']:.0f}%, MaxDD: {best_sharpe['max_dd']:.0f}%")

print(f"\n🛡️ Best Drawdown: Buy < {best_dd['buy_thresh']:.2f}, Sell > {best_dd['sell_thresh']:.2f}")
print(f"   MaxDD: {best_dd['max_dd']:.0f}%, Return: {best_dd['total_return']:.0f}%, Sharpe: {best_dd['sharpe']:.2f}")

In [ ]:
# Heatmap of results
import matplotlib.colors as mcolors

# Create pivot tables
df_grid = pd.DataFrame(grid_results)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, title in zip(axes, ['cagr', 'sharpe', 'max_dd'], 
                              ['CAGR (%)', 'Sharpe Ratio', 'Max Drawdown (%)']):
    pivot = df_grid.pivot_table(values=metric, index='buy_thresh', columns='sell_thresh')
    
    if metric == 'max_dd':
        cmap = 'RdYlGn'  # Red = bad (negative), Green = good (less negative)
    else:
        cmap = 'RdYlGn'
    
    im = ax.imshow(pivot.values, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{x:.1f}' for x in pivot.columns], rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{x:.1f}' for x in pivot.index])
    ax.set_xlabel('Sell Threshold')
    ax.set_ylabel('Buy Threshold')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)

plt.suptitle('Threshold Optimization Heatmaps', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Final Recommendation

In [ ]:
# Current market state
latest = df.iloc[-1]

print("\n" + "#"*70)
print("TRIGGER STRATEGY - CURRENT STATUS")
print("#"*70)

print(f"\n📅 Date: {latest.name.date()}")
print(f"💰 BTC Price: ${latest['price']:,.0f}")
print(f"📊 Composite Score: {latest['composite']:+.2f}")

# Using best Sharpe thresholds
buy_t = best_sharpe['buy_thresh']
sell_t = best_sharpe['sell_thresh']

print(f"\n🎯 Optimal Thresholds (Best Sharpe):")
print(f"   Buy when score < {buy_t:.2f}")
print(f"   Sell when score > {sell_t:.2f}")

if latest['composite'] < buy_t:
    print(f"\n🟢 SIGNAL: BUY - Go to 100% allocation!")
    print(f"   Score {latest['composite']:.2f} < {buy_t:.2f} threshold")
elif latest['composite'] > sell_t:
    print(f"\n🔴 SIGNAL: SELL - Reduce to 25% allocation!")
    print(f"   Score {latest['composite']:.2f} > {sell_t:.2f} threshold")
else:
    print(f"\n🟡 SIGNAL: HOLD - Maintain current position")
    print(f"   Score {latest['composite']:.2f} between thresholds")
    print(f"   Distance to BUY: {latest['composite'] - buy_t:.2f}")
    print(f"   Distance to SELL: {sell_t - latest['composite']:.2f}")

## Summary

**Trigger Strategy Benefits:**
1. ✅ Low turnover - only trades at extremes
2. ✅ High conviction - acts on clear signals
3. ✅ Simple rules - easy to follow
4. ✅ Reduced drawdowns vs HODL

**Trade-offs:**
1. ⚠️ May underperform HODL in strong bull markets
2. ⚠️ Requires discipline to hold during uncertainty
3. ⚠️ Threshold optimization is backward-looking

**Recommended Approach:**
- **Conservative:** Buy < -1.5, Sell > 2.0 (fewer trades, higher conviction)
- **Balanced:** Buy < -1.0, Sell > 1.5 (good risk/reward)
- **Aggressive:** Buy < -0.75, Sell > 1.25 (more active)